# Simple Pendulum: Physics Validation

This notebook explores the simple pendulum model in BLUR, validating its physics
against known analytical results.

**What we will demonstrate:**
1. Period validation against the small-angle formula: $T = 2\pi\sqrt{L/g}$
2. Energy conservation during simulation

**Key concepts:**
- The simple pendulum is a classic benchmark for physics simulators
- State vector: $[\theta, \dot{\theta}]$ (angle and angular velocity)
- Equation of motion: $\ddot{\theta} = -\frac{g}{L}\sin(\theta)$

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
import jax
import jax.numpy as jnp

from fmd.simulator import SimplePendulum, simulate
from fmd.simulator.params import SimplePendulumParams, PENDULUM_1M

# Create a 1-meter pendulum using the preset
pendulum = SimplePendulum(PENDULUM_1M)

print(f"Pendulum length: {pendulum.length} m")
print(f"Gravity: {pendulum.g} m/s^2")
print(f"State names: {pendulum.state_names}")
print(f"JAX backend: {jax.default_backend()}")

## 1. Period Validation

For small oscillations, the pendulum behaves like a simple harmonic oscillator.
The period is given by:

$$T = 2\pi\sqrt{\frac{L}{g}}$$

Let's run a simulation and measure the period from the trajectory.

In [ ]:
# Small initial angle for small-angle approximation validity
theta0 = 0.05  # radians (~3 degrees)
initial_state = jnp.array([theta0, 0.0])  # [theta, theta_dot]

# Theoretical period
T_theory = float(pendulum.period_small_angle())
print(f"Theoretical period (small angle): {T_theory:.4f} s")

# Simulate for several periods
n_periods = 5
duration = n_periods * T_theory
dt = 0.001  # Fine timestep for accuracy

result = simulate(pendulum, initial_state, dt=dt, duration=duration)
print(f"Simulated {duration:.2f} s ({n_periods} periods)")
print(f"Number of timesteps: {len(result.times)}")

In [ ]:
# Extract trajectory data
times = np.array(result.times)
theta = np.array(result.states[:, 0])  # angle
theta_dot = np.array(result.states[:, 1])  # angular velocity

# Plot the oscillation
fig, axes = plt.subplots(2, 1, figsize=(10, 6), sharex=True)

axes[0].plot(times, np.degrees(theta), 'b-', linewidth=1)
axes[0].set_ylabel('Angle (degrees)')
axes[0].set_title('Pendulum Oscillation')
axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='k', linewidth=0.5)

axes[1].plot(times, np.degrees(theta_dot), 'r-', linewidth=1)
axes[1].set_xlabel('Time (s)')
axes[1].set_ylabel('Angular velocity (deg/s)')
axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='k', linewidth=0.5)

# Mark theoretical period locations
for i in range(n_periods + 1):
    for ax in axes:
        ax.axvline(i * T_theory, color='g', linestyle='--', alpha=0.5, linewidth=0.8)

plt.tight_layout()
plt.show()

In [ ]:
# Measure period from zero crossings (theta going from positive to negative)
# These occur once per period at the equilibrium position

zero_crossings = []
for i in range(1, len(theta)):
    if theta[i-1] > 0 and theta[i] <= 0:
        # Linear interpolation to find precise crossing time
        t_cross = times[i-1] + (times[i] - times[i-1]) * (theta[i-1] / (theta[i-1] - theta[i]))
        zero_crossings.append(t_cross)

print(f"Found {len(zero_crossings)} zero crossings (positive-to-negative)")
print(f"Crossing times: {[f'{t:.4f}' for t in zero_crossings]}")

# Measure periods between consecutive crossings
measured_periods = [zero_crossings[i+1] - zero_crossings[i] for i in range(len(zero_crossings)-1)]
print(f"\nMeasured periods: {[f'{p:.4f}' for p in measured_periods]}")

In [ ]:
# Compare measured vs theoretical period
T_measured = np.mean(measured_periods)
T_std = np.std(measured_periods)
error_percent = 100 * abs(T_measured - T_theory) / T_theory

print("Period Validation Results")
print("=" * 40)
print(f"Theoretical period:  {T_theory:.6f} s")
print(f"Measured period:     {T_measured:.6f} s (+/- {T_std:.6f})")
print(f"Absolute error:      {abs(T_measured - T_theory)*1000:.4f} ms")
print(f"Relative error:      {error_percent:.4f}%")
print()

if error_percent < 0.1:
    print("PASS: Period matches theory within 0.1%")
else:
    print(f"Note: Error of {error_percent:.2f}% may be due to finite timestep")

## 2. Energy Conservation

For a conservative system like the ideal pendulum (no friction), total mechanical
energy should be conserved. The energy components are:

**Kinetic Energy:**
$$KE = \frac{1}{2} m L^2 \dot{\theta}^2$$

**Potential Energy** (with zero at the lowest point):
$$PE = m g L (1 - \cos\theta)$$

**Total Energy:**
$$E = KE + PE = \text{constant}$$

The pendulum model uses unit mass ($m=1$), so the formulas simplify to:
- $KE = \frac{1}{2} L^2 \dot{\theta}^2$
- $PE = g L (1 - \cos\theta)$

In [ ]:
# Run a longer simulation with a larger initial angle
theta0_large = 0.5  # radians (~29 degrees)
initial_large = jnp.array([theta0_large, 0.0])

# Simulate for 10 seconds
result_energy = simulate(pendulum, initial_large, dt=0.001, duration=10.0)

times_e = np.array(result_energy.times)
states_e = np.array(result_energy.states)
theta_e = states_e[:, 0]
theta_dot_e = states_e[:, 1]

print(f"Initial angle: {np.degrees(theta0_large):.1f} degrees")
print(f"Simulation duration: 10 s")
print(f"Timesteps: {len(times_e)}")

In [ ]:
# Calculate energy components at each timestep
L = pendulum.length
g = pendulum.g

# Unit mass (m=1) as used in the SimplePendulum model
KE = 0.5 * L**2 * theta_dot_e**2
PE = g * L * (1 - np.cos(theta_e))
total_energy = KE + PE

# Initial energy
E0 = total_energy[0]

print(f"Initial kinetic energy:   {KE[0]:.6f} J")
print(f"Initial potential energy: {PE[0]:.6f} J")
print(f"Initial total energy:     {E0:.6f} J")

In [ ]:
# Plot energy over time
fig, axes = plt.subplots(2, 1, figsize=(10, 8))

# Top plot: Energy components
ax1 = axes[0]
ax1.plot(times_e, KE, 'b-', label='Kinetic Energy', linewidth=1, alpha=0.8)
ax1.plot(times_e, PE, 'r-', label='Potential Energy', linewidth=1, alpha=0.8)
ax1.plot(times_e, total_energy, 'k-', label='Total Energy', linewidth=2)
ax1.set_ylabel('Energy (J)')
ax1.set_title('Pendulum Energy Over Time')
ax1.legend(loc='right')
ax1.grid(True, alpha=0.3)

# Bottom plot: Energy deviation from initial
ax2 = axes[1]
energy_error = (total_energy - E0) / E0 * 100  # Percent deviation
ax2.plot(times_e, energy_error, 'g-', linewidth=1)
ax2.set_xlabel('Time (s)')
ax2.set_ylabel('Energy Error (%)')
ax2.set_title('Energy Conservation Error')
ax2.grid(True, alpha=0.3)
ax2.axhline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

In [ ]:
# Quantify energy conservation
energy_drift = total_energy[-1] - E0
energy_drift_percent = 100 * energy_drift / E0
max_error = np.max(np.abs(total_energy - E0))
max_error_percent = 100 * max_error / E0

print("Energy Conservation Results")
print("=" * 40)
print(f"Initial energy:      {E0:.8f} J")
print(f"Final energy:        {total_energy[-1]:.8f} J")
print(f"Energy drift:        {energy_drift:.2e} J ({energy_drift_percent:.4f}%)")
print(f"Max deviation:       {max_error:.2e} J ({max_error_percent:.4f}%)")
print()

if max_error_percent < 0.01:
    print("PASS: Energy conserved within 0.01%")
elif max_error_percent < 0.1:
    print("PASS: Energy conserved within 0.1%")
else:
    print(f"Note: Energy drift of {max_error_percent:.2f}% detected")

In [ ]:
# Use the built-in energy method for verification
energies_builtin = jax.vmap(pendulum.energy)(result_energy.states)
energies_builtin = np.array(energies_builtin)

# Compare our manual calculation with built-in
difference = np.max(np.abs(total_energy - energies_builtin))
print(f"Max difference between manual and built-in energy: {difference:.2e}")
print("(Should be essentially zero, confirming our formulas match the model)")

## 3. Phase Space Portrait

A phase portrait shows the relationship between position (angle) and velocity.
For a conservative system, the trajectory forms a closed orbit.

In [ ]:
# Plot phase portrait
fig, ax = plt.subplots(figsize=(8, 6))

ax.plot(np.degrees(theta_e), np.degrees(theta_dot_e), 'b-', linewidth=1)
ax.plot(np.degrees(theta_e[0]), np.degrees(theta_dot_e[0]), 'go', markersize=10, label='Start')
ax.plot(np.degrees(theta_e[-1]), np.degrees(theta_dot_e[-1]), 'r^', markersize=10, label='End')

ax.set_xlabel('Angle (degrees)')
ax.set_ylabel('Angular Velocity (deg/s)')
ax.set_title('Phase Portrait (should be a closed orbit)')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_aspect('equal', adjustable='datalim')
ax.axhline(0, color='k', linewidth=0.5)
ax.axvline(0, color='k', linewidth=0.5)

plt.tight_layout()
plt.show()

# How close is the final state to the initial state?
state_error = np.linalg.norm(states_e[-1] - states_e[0])
print(f"Distance from start to end in phase space: {state_error:.6f}")
print("(Close to zero indicates good energy conservation and numerical stability)")

## Summary

This notebook validated the BLUR simple pendulum model by:

1. **Period validation**: The simulated period matches the theoretical small-angle
   formula $T = 2\pi\sqrt{L/g}$ to within a fraction of a percent.

2. **Energy conservation**: Total mechanical energy (kinetic + potential) remains
   constant throughout the simulation, demonstrating the accuracy of the RK4 integrator.

3. **Phase portrait**: The trajectory forms a closed orbit in phase space, confirming
   the conservative nature of the dynamics.

These results build confidence that the BLUR simulator correctly implements the
underlying physics, which is essential before moving on to more complex systems.